In [ ]:
# Project FOD points to match the SEFM coordinate system
import arcpy
import os

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
fod_gdb        = os.path.join(base_dir, "data", "7thED_FPA_FOD_20260515.gdb")

events         = os.path.join(classifire_gdb, "SEFM_events_94_24")
fod_gcs        = os.path.join(fod_gdb, "Fires") # The raw GCS layer
fod_pcs        = os.path.join(classifire_gdb, "FPA_FOD_2026_projected")

print("Extracting target spatial reference from SEFM_events...")
target_sr = arcpy.Describe(events).spatialReference
print(f"Target Projection: {target_sr.name}")

print("Projecting FOD points... ")
# Run the projection tool
arcpy.management.Project(
    in_dataset=fod_gcs,
    out_dataset=fod_pcs,
    out_coor_system=target_sr
)

print(f"Projected points saved to: {fod_pcs}")

In [ ]:
# Clip the projected FOD points to the dedicated SE FireMap extent
import arcpy
import os

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
fod_pcs        = os.path.join(classifire_gdb, "FPA_FOD_2026_projected")
sefm_extent    = os.path.join(classifire_gdb, "extent_Dissolved")
fod_clipped    = os.path.join(classifire_gdb, "FPA_FOD_2026_SE_clipped")

print("Clipping projected FOD points to the extent_Dissolved boundary...")

# Run the clip tool using your dedicated extent layer
arcpy.analysis.Clip(
    in_features=fod_pcs,
    clip_features=sefm_extent,
    out_feature_class=fod_clipped
)

print(f"Clipped regional points saved to: {fod_clipped}")

In [ ]:
# Filter FOD points to the years 1994-2024 and save as FPA_FOD_94_24
import arcpy
import os

# --- PATHS ---
classifire_gdb = r"C:\GIS\ClassiFIRE\Project\ClassiFIRE.gdb"
fod_clipped    = os.path.join(classifire_gdb, "FPA_FOD_2026_SE_clipped")
fod_94_24      = os.path.join(classifire_gdb, "FPA_FOD_94_24")

# Enable overwriting so you can run this cleanly
arcpy.env.overwriteOutput = True

print("Filtering FOD points to years 1994-2024...")

# Create an in-memory feature layer to apply the attribute selection
temp_layer = "fod_temp_year_layer"
arcpy.management.MakeFeatureLayer(fod_clipped, temp_layer)

# Define the SQL expression for the year range based on FIRE_YEAR
sql_expression = "FIRE_YEAR >= 1994 AND FIRE_YEAR <= 2024"

# Apply the selection
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=temp_layer,
    selection_type="NEW_SELECTION",
    where_clause=sql_expression
)

# Save the selected features to the new feature class
arcpy.management.CopyFeatures(temp_layer, fod_94_24)

# Clean up the temporary in-memory layer
arcpy.management.Delete(temp_layer)

print(f"Filtered baseline points saved to: {fod_94_24}")

In [ ]:
# Add area_ha field and convert FIRE_SIZE (acres) to hectares
import arcpy
import os

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
fod_filtered   = os.path.join(classifire_gdb, "FPA_FOD_94_24")

print("Checking fields in FPA_FOD_94_24...")
existing_fields = [f.name for f in arcpy.ListFields(fod_filtered)]

# --- 1. ADD FIELD ---
if "area_ha" not in existing_fields:
    print("Adding area_ha field...")
    arcpy.management.AddField(
        in_table=fod_filtered,
        field_name="area_ha",
        field_type="DOUBLE",
        field_alias="area_ha"
    )
    print("Field 'area_ha' added successfully.")
else:
    print("Field 'area_ha' already exists.")

# --- 2. CALCULATE HECTARES ---
print("Calculating hectares from FIRE_SIZE...")

# Using CalculateField with an expression block
# 1 acre = 0.40468564224 hectares
expression = "!FIRE_SIZE! * 0.40468564224"

arcpy.management.CalculateField(
    in_table=fod_filtered,
    field="area_ha",
    expression=expression,
    expression_type="PYTHON3"
)

print("'area_ha' calculation complete.")

In [ ]:
# Filter out fires smaller than 0.809 hectares (2 acres)
import arcpy
import os

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
fod_baseline   = os.path.join(classifire_gdb, "FPA_FOD_94_24")
fod_large      = os.path.join(classifire_gdb, "FPA_FOD_94_24_large")

# Enable overwriting for safety
arcpy.env.overwriteOutput = True

print("Extracting fires >= 0.809 ha to a new layer...")

# Create an in-memory feature layer to apply the size selection
temp_layer = "fod_large_filter_layer"
arcpy.management.MakeFeatureLayer(fod_baseline, temp_layer)

# Define the SQL expression to grab fires 2 acres or larger
sql_expression = "area_ha >= 0.809"

# Apply the selection
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=temp_layer,
    selection_type="NEW_SELECTION",
    where_clause=sql_expression
)

# Save the selected features to a brand-new feature class
arcpy.management.CopyFeatures(temp_layer, fod_large)

# Clean up the temporary in-memory layer
arcpy.management.Delete(temp_layer)

print(f"Size-filtered points saved to: {fod_large}")
print("baseline 'FPA_FOD_94_24' remains completely intact.")